In [1]:
import nltk
from nltk.stem.porter import PorterStemmer
import numpy as np

In [2]:
nltk.download('punkt')
stemmer = PorterStemmer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# NLTK Utils

In [3]:
def tokenize(sentence):
    return nltk.word_tokenize(sentence)

def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, all_words):
    tokenized_sentence = [stem(w) for w in tokenized_sentence]
    bag = np.zeros(len(all_words),dtype = np.float32)
    for index,w in enumerate(all_words):
        if w in tokenized_sentence:
            bag[index] = 1.0
    return bag

# Model

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [5]:
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

# Training

In [10]:
import json
with open('intents.json','r') as f:
    intents = json.load(f)

In [13]:
all_words = []
tags = []
xy = []
for intent in intents['intents']:
    tag = intent['tag']
    tags.append(tag)
    for pattern in intent['patterns']:
        w = tokenize(pattern)
        all_words.extend(w)
        xy.append((w,tag))

In [8]:
ignore_words = ['?','!', '.',',','{','}']

## Εδώ λογικά πρέπει να μπούνε τα δεδομένα μου από τη βάση i guess

In [14]:
all_words = [stem(w) for w in all_words if w not in ignore_words]
all_words = sorted(set(all_words))

In [15]:
X_train = []
Y_train = []
for (pattern_sentence, tag) in xy:
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)

    label = tags.index(tag)
    Y_train.append(label)

X_train = np.array(X_train)
Y_train = np.array(Y_train)

In [16]:
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = X_train
        self.y_data = Y_train

    def __getitem__(self,index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples



In [17]:
batch_size = 8
hidden_size = 8
output_size = len(tags)
input_size = len(all_words)
learning_rate = 0.001
epochs = 1000

In [18]:
dataset = ChatDataset()
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, num_workers = 0)

In [19]:
model = NeuralNet(input_size, hidden_size, output_size)

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [21]:
for epoch in range(epochs):
    for (words, labels) in train_loader:
        outputs = model(words)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'epoch{epoch + 1}/{epochs}, loss={loss.item():.4f}')


epoch100/1000, loss=0.6742
epoch200/1000, loss=0.7218
epoch300/1000, loss=0.0255
epoch400/1000, loss=0.0094
epoch500/1000, loss=0.0022
epoch600/1000, loss=0.0070
epoch700/1000, loss=0.0019
epoch800/1000, loss=0.0005
epoch900/1000, loss=0.0009
epoch1000/1000, loss=0.0012


In [25]:
data = {
    "model_state" : model.state_dict(),
    "input_size": input_size,
    "output_size": output_size,
    "hidden_size": hidden_size,
    "all_words": all_words,
    "tags": tags
}

FILE = "./data.pth"
torch.save(data,FILE)

In [27]:
# Chat

In [26]:
import random

In [ ]:
with open('intents.json','r') as f:
  intents = json.load(f)

data = torch.load(FILE)

input_size = data["input_size"]
hidden_size = data["hidden_size"]
output_size = data["output_size"]
all_words = data["all_words"]
tags = data["tags"]
model_state = data["model_state"]

model.load_state_dict(model_state)
model.eval()

bot_name = "Tao"
print("Let's chat! To stop type 'quit'!")
while(True):
  sentence = input('You: ')
  if sentence == "quit":
    break
    sentence = tokenize(sentence)
    X = bag_of_words(sentence, all_words)
    X = X.reshape(1, X.shape[0])
    X = torch.from_numpy()

    output = model(X)
    _, predicted = torch.max(output, dim=1)
    tag = tags[predicted.item()]

    probs = torch.softmax(output, dim=1)
    prob = probs[0][predicted.item()]

    if prob.item() > 0.75:
      for intent in intents["intents"]:
        if tag == intent['tag']:
          print(f'{bot_name}: {random.choice(intent["responses"])}')
    else:
      print(f'{bot_name}: I don"t get it!')


Let's chat! To stop type 'quit'!
You: hi
You: hello
You: movies
You: dwda
You: 
You: 
You: 
You: dwa\n
